In [1]:
DATA_DIR = "dataset/"
MODEL_DIR = "model/"
CHECKPOINT_DIR = "epoch_model/"
RESULT_DIR = "result/"

In [2]:
USERS_CSV = DATA_DIR + "users.csv"
TRAIN_CSV = DATA_DIR + "train.csv"
PREDICT_CSV = DATA_DIR + "test.csv"

# Define Model

In [3]:
MODEL_NAME = "sklearn_RandomForestClassifier_maxdepth20"


from sklearn.ensemble import RandomForestClassifier


model = RandomForestClassifier(max_depth=20, random_state=0)

# Import Data

In [4]:
import pandas


raw_users = pandas.read_csv(USERS_CSV)
raw_train = pandas.read_csv(TRAIN_CSV)
raw_predict = pandas.read_csv(PREDICT_CSV)

In [5]:
from datetime import datetime


def get_unix(s):
    _temp = datetime.strptime(s[:-6], "%Y-%m-%d %H:%M:%S")
    return int((_temp - datetime(1970,1,1)).total_seconds())

In [6]:
def convert_nan(x):
    try:
        return int(x)
    except:
        return 1000 # impossible value

In [7]:
domain_map = {
    "@gmail.com"      : 0,
    "@yahoo.com"      : 1,
    "@hotmail.com"    : 2,
    "other"           : 3,
    "@icloud.com"     : 4,
    "@qq.com"         : 5,
    "@outlook.com"    : 6,
    "@ymail.com"      : 7,
    "@live.com"       : 8,
    "@163.com"        : 9,
    "@rocketmail.com" : 10,
}

In [8]:
import numpy
from sklearn.preprocessing import StandardScaler


def normalize_scaler(a, b):
    a = a.to_numpy().reshape(-1, 1)
    b = b.to_numpy().reshape(-1, 1)
    scaler = StandardScaler()
    return scaler.fit_transform(a), scaler.transform(b)

In [9]:
import numpy
from sklearn.preprocessing import OneHotEncoder


def normalize_ohe(a, b):
    return a, b

In [10]:
users_data = raw_users.copy(deep=True)

users_data["attr_1"] = users_data["attr_1"].apply(convert_nan)
users_data["attr_2"] = users_data["attr_2"].apply(convert_nan)
users_data["attr_3"] = users_data["attr_3"].apply(convert_nan)
users_data["age"] = users_data["age"].apply(convert_nan)

In [11]:
train_data = pandas.merge(raw_train.copy(deep=True), users_data, on="user_id", how="left")
predict_data = pandas.merge(raw_predict.copy(deep=True), users_data, on="user_id", how="left")

train_data["grass_date"] = train_data["grass_date"].apply(get_unix)
predict_data["grass_date"] = predict_data["grass_date"].apply(get_unix)

train_data["last_open_day"] = train_data["last_open_day"].apply(convert_nan)
predict_data["last_open_day"] = predict_data["last_open_day"].apply(convert_nan)

train_data["last_login_day"] = train_data["last_login_day"].apply(convert_nan)
predict_data["last_login_day"] = predict_data["last_login_day"].apply(convert_nan)

train_data["last_checkout_day"] = train_data["last_checkout_day"].apply(convert_nan)
predict_data["last_checkout_day"] = predict_data["last_checkout_day"].apply(convert_nan)

train_data["domain"] = train_data["domain"].replace(domain_map)
predict_data["domain"] = predict_data["domain"].replace(domain_map)

train_data["country_code"], predict_data["country_code"] = normalize_ohe(train_data["country_code"], predict_data["country_code"])
train_data["grass_date"], predict_data["grass_date"] = normalize_ohe(train_data["grass_date"], predict_data["grass_date"])
train_data["subject_line_length"], predict_data["subject_line_length"] = normalize_scaler(train_data["subject_line_length"], predict_data["subject_line_length"])
train_data["last_open_day"], predict_data["last_open_day"] = normalize_scaler(train_data["last_open_day"], predict_data["last_open_day"])
train_data["last_login_day"], predict_data["last_login_day"] = normalize_scaler(train_data["last_login_day"], predict_data["last_login_day"])
train_data["last_checkout_day"], predict_data["last_checkout_day"] = normalize_scaler(train_data["last_checkout_day"], predict_data["last_checkout_day"])
train_data["open_count_last_10_days"], predict_data["open_count_last_10_days"] = normalize_scaler(train_data["open_count_last_10_days"], predict_data["open_count_last_10_days"])
train_data["open_count_last_30_days"], predict_data["open_count_last_30_days"] = normalize_scaler(train_data["open_count_last_30_days"], predict_data["open_count_last_30_days"])
train_data["open_count_last_60_days"], predict_data["open_count_last_60_days"] = normalize_scaler(train_data["open_count_last_60_days"], predict_data["open_count_last_60_days"])
train_data["login_count_last_10_days"], predict_data["login_count_last_10_days"] = normalize_scaler(train_data["login_count_last_10_days"], predict_data["login_count_last_10_days"])
train_data["login_count_last_30_days"], predict_data["login_count_last_30_days"] = normalize_scaler(train_data["login_count_last_30_days"], predict_data["login_count_last_30_days"])
train_data["login_count_last_60_days"], predict_data["login_count_last_60_days"] = normalize_scaler(train_data["login_count_last_60_days"], predict_data["login_count_last_60_days"])
train_data["checkout_count_last_10_days"], predict_data["checkout_count_last_10_days"] = normalize_scaler(train_data["checkout_count_last_10_days"], predict_data["checkout_count_last_10_days"])
train_data["checkout_count_last_30_days"], predict_data["checkout_count_last_30_days"] = normalize_scaler(train_data["checkout_count_last_30_days"], predict_data["checkout_count_last_30_days"])
train_data["checkout_count_last_60_days"], predict_data["checkout_count_last_60_days"] = normalize_scaler(train_data["checkout_count_last_60_days"], predict_data["checkout_count_last_60_days"])
train_data["attr_1"], predict_data["attr_1"] = normalize_ohe(train_data["attr_1"], predict_data["attr_1"])
train_data["attr_2"], predict_data["attr_2"] = normalize_ohe(train_data["attr_2"], predict_data["attr_2"])
train_data["attr_3"], predict_data["attr_3"] = normalize_ohe(train_data["attr_3"], predict_data["attr_3"])
train_data["age"], predict_data["age"] = normalize_ohe(train_data["age"], predict_data["age"])
train_data["domain"], predict_data["domain"] = normalize_ohe(train_data["domain"], predict_data["domain"])

In [12]:
from sklearn.model_selection import train_test_split


X_train_dataset, X_test_dataset, y_train_dataset, y_test_dataset = train_test_split(
    train_data.loc[:, (train_data.columns != "open_flag") & ((train_data.columns != "row_id"))],
    train_data["open_flag"],
    test_size=0.2,
    random_state=17
)

# Train Model

In [13]:
model = model.fit(X_train_dataset, y_train_dataset)

# Accuracy Test

In [14]:
import numpy
from sklearn import metrics


y_pred = model.predict(X_test_dataset)

print("Acc : {0:.5f}".format(metrics.accuracy_score(y_test_dataset, y_pred)))
print("MAE : {0:.5f}".format(metrics.mean_absolute_error(y_test_dataset, y_pred)))
print("MSE : {0:.5f}".format(metrics.mean_squared_error(y_test_dataset, y_pred)))
print("RMSE: {0:.5f}".format(numpy.sqrt(metrics.mean_squared_error(y_test_dataset, y_pred))))
print("MCC : {0:.5f}".format(metrics.matthews_corrcoef(y_test_dataset, y_pred)))

Acc : 0.88754
MAE : 0.11246
MSE : 0.11246
RMSE: 0.33534
MCC : 0.51772


# Predict

In [15]:
predict = predict_data.copy(deep=True)


predict["open_flag"] = model.predict(predict.loc[:, predict.columns != "row_id"])

# Save Result

In [16]:
result = predict.copy(deep=True)


result = result.drop(
    columns=[
        "country_code",
        "grass_date",
        "user_id",
        "subject_line_length",
        "last_open_day",
        "last_login_day",
        "last_checkout_day",
        "open_count_last_10_days",
        "open_count_last_30_days",
        "open_count_last_60_days",
        "login_count_last_10_days",
        "login_count_last_30_days",
        "login_count_last_60_days",
        "checkout_count_last_10_days",
        "checkout_count_last_30_days",
        "checkout_count_last_60_days",
        "attr_1",
        "attr_2",
        "attr_3",
        "age",
        "domain",
    ]
)

In [17]:
result.to_csv(RESULT_DIR + MODEL_NAME + "_result.csv", index=False)